# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinukondablessena/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: Learned model versus baseline

The research material reports that the random forest achieved higher observed performance than the fixed baseline, including Precision@50 of 0.740 compared with 0.240 on the starter validation setup. This suggests that the learned model produced a stronger ranking on that measured validation set.

**Methodology question:** Does the validation design and label definition fully support interpreting this difference as evidence that the learned model is better for the intended decision-support task?

### Finding 2: Decline label

The starter modeling workflow defines `is_declining_label` from `trend_direction == "down"`. This is a current-window proxy label rather than a direct future outcome.

**Methodology question:** How closely does this current-window label represent the future outcome we actually want to identify, and would a future-looking label change the observed model performance?

These are constructive methodology questions rather than judgments about the research. They focus on whether the label and validation design support the scope of the claims.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Validation approach

I compare the Week-5 Random Forest using a standard random train/test split with a client-grouped split. The grouped split keeps all rows from a client in either training or testing, which is a stricter check because pages from the same client can share patterns.

The goal is to see whether the measured model performance changes when the model is evaluated on clients that were not represented in training.

I will treat the results as observed validation performance and decision-support evidence, not as proof of future business impact.

In [16]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFolders in /content:")
print(os.listdir("/content")[:30])

Current folder:
/content/flyrank-ml-internship

Folders in /content:
['.config', 'flyrank-ml-internship', 'sample_data']


In [17]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# Load prepared dataset
DATA_PATH = "/content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv"

df = pd.read_csv(DATA_PATH)

TARGET = "is_declining_label"
GROUP = "client_id"

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d",
    "log_ai_sessions_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# Check columns
missing = [c for c in [TARGET, GROUP] + FEATURES if c not in df.columns]

print("Dataset shape:", df.shape)
print("Missing required columns:", missing)

if missing:
    raise ValueError(f"Missing columns: {missing}")

X = df[FEATURES].copy()
y = df[TARGET].astype(int)
groups = df[GROUP]

print("Target rate:", round(y.mean(), 4))
print("Number of clients:", groups.nunique())


# Preprocessing
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), CATEGORICAL_FEATURES)
])


def make_model():
    return Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])


def precision_at_k(y_true, scores, k=50):
    order = np.argsort(scores)[::-1][:k]
    return np.mean(np.asarray(y_true)[order])


# =========================================================
# BEFORE: Random split
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

before_model = make_model()
before_model.fit(X_train, y_train)

before_prob = before_model.predict_proba(X_test)[:, 1]

before_auc = roc_auc_score(y_test, before_prob)
before_ap = average_precision_score(y_test, before_prob)
before_p50 = precision_at_k(y_test.values, before_prob, 50)

print("\n=== BEFORE: Random split ===")
print("ROC-AUC:", round(before_auc, 4))
print("Average Precision:", round(before_ap, 4))
print("Precision@50:", round(before_p50, 4))


# =========================================================
# AFTER: Client-grouped split
# =========================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

after_model = make_model()
after_model.fit(X_train_group, y_train_group)

after_prob = after_model.predict_proba(X_test_group)[:, 1]

after_auc = roc_auc_score(y_test_group, after_prob)
after_ap = average_precision_score(y_test_group, after_prob)
after_p50 = precision_at_k(
    y_test_group.values,
    after_prob,
    50
)

print("\n=== AFTER: Client-grouped split ===")
print("ROC-AUC:", round(after_auc, 4))
print("Average Precision:", round(after_ap, 4))
print("Precision@50:", round(after_p50, 4))
print("Training clients:", groups_train.nunique())
print("Testing clients:", groups_test.nunique())
print("Client overlap:", len(set(groups_train) & set(groups_test)))


# =========================================================
# Comparison
# =========================================================

comparison = pd.DataFrame({
    "validation": [
        "Random split",
        "Client-grouped split"
    ],
    "ROC-AUC": [
        before_auc,
        after_auc
    ],
    "Average Precision": [
        before_ap,
        after_ap
    ],
    "Precision@50": [
        before_p50,
        after_p50
    ]
})

print("\n=== BEFORE / AFTER COMPARISON ===")
display(comparison.round(4))

Dataset shape: (30000, 52)
Missing required columns: []
Target rate: 0.5421
Number of clients: 32

=== BEFORE: Random split ===
ROC-AUC: 0.7682
Average Precision: 0.7807
Precision@50: 0.94

=== AFTER: Client-grouped split ===
ROC-AUC: 0.6044
Average Precision: 0.5998
Precision@50: 0.72
Training clients: 25
Testing clients: 7
Client overlap: 0

=== BEFORE / AFTER COMPARISON ===


,validation,ROC-AUC,Average Precision,Precision@50
0,Random split,0.7682,0.7807,0.94
1,Client-grouped split,0.6044,0.5998,0.72


### Before / after result

The random split produced observed ROC-AUC of 0.7682, average precision of 0.7807, and Precision@50 of 0.94. Under the client-grouped split, the measured values decreased to ROC-AUC 0.6044, average precision 0.5998, and Precision@50 0.72.

The grouped split kept all 32 clients separated between training and testing, with 25 clients in training, 7 clients in testing, and zero client overlap.

This comparison shows that the validation design materially affects the measured model performance. The random split produced more optimistic results than the client-grouped evaluation. Therefore, I will use the client-grouped result as the more conservative validation result for this audit.

I interpret these results as observed and measured decision-support performance, not as evidence of future business impact or causality.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [18]:
# =========================================================
# 3. LEAKAGE AUDIT
# =========================================================

print("=== LEAKAGE AUDIT ===")

# Fields that must not be model features
risky_fields = [
    TARGET,
    GROUP,
    "trend_direction",
    "trend_pct"
]

print("\nFinal feature count:", len(FEATURES))

print("\nChecking for target/group/risky fields:")
for field in risky_fields:
    print(f"{field}: {'FOUND' if field in FEATURES else 'NOT FOUND'}")

# Check for exact overlap
leakage_matches = [field for field in risky_fields if field in FEATURES]

print("\nPotential leakage fields found in FEATURES:")
print(leakage_matches)

# Check that target is not in X
print("\nTarget in X:", TARGET in X.columns)

# Check that client_id is not in X
print("Client ID in X:", GROUP in X.columns)

# Final result
if len(leakage_matches) == 0 and TARGET not in X.columns and GROUP not in X.columns:
    print("\nPASS: No direct target, grouping, or known risky fields found in final features.")
else:
    print("\nREVIEW REQUIRED: Potential leakage detected.")

=== LEAKAGE AUDIT ===

Final feature count: 26

Checking for target/group/risky fields:
is_declining_label: NOT FOUND
client_id: NOT FOUND
trend_direction: NOT FOUND
trend_pct: NOT FOUND

Potential leakage fields found in FEATURES:
[]

Target in X: False
Client ID in X: False

PASS: No direct target, grouping, or known risky fields found in final features.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim Rewrite

### Original claim

The Random Forest model performs well at identifying declining content and can be used to improve content decisions.

### Safer claim

Under the client-grouped validation setup, the Random Forest achieved an observed ROC-AUC of 0.6044, average precision of 0.5998, and Precision@50 of 0.72. These results provide measured, directional evidence that the model may be useful for ranking content for decision support on clients not represented in training. They do not establish future business impact, causality, or production performance.


In [19]:
# Section 4: Claim rewrite
# Report the conservative validation result used in the claim.

print("Conservative validation metrics:")
print("ROC-AUC:", round(after_auc, 4))
print("Average Precision:", round(after_ap, 4))
print("Precision@50:", round(after_p50, 4))

Conservative validation metrics:
ROC-AUC: 0.6044
Average Precision: 0.5998
Precision@50: 0.72


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.